# YOLOv8 — Benchmark on OOD Dataset (22 classes)

**Default:** no training — rebuilds the benchmark table from existing `../runs/detect/<variant>_ood*/weights/best.pt` (newest run per variant). Set `RUN_TRAINING = True` in the config cell only when you want a full train.

Compare:
- **mAP@0.5** and **mAP@0.5:0.95**
- **Precision**, **Recall**, **F1 Score**
- **Inference speed** (ms/img)
- **Model size** (MB) and **parameters** (M)
- **Per-class AP**

In [7]:
# Order: GPU check → config → function cells → **benchmark / build CSV** → tables & plots.


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


: 

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os


def _safe_cuda_empty_cache():
    """Avoid crashing the notebook if CUDA is already in an error state."""
    if not torch.cuda.is_available():
        return
    try:
        torch.cuda.synchronize()
    except Exception:
        pass
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass


# False = only evaluate saved weights and write CSV (for tables). True = full training loop (slow).
RUN_TRAINING = False
# When RUN_TRAINING is False: short names whose newest best.pt is picked under ../runs/detect/
BENCHMARK_VARIANTS = ["yolov8n", "yolov8s", "yolov8m"]

# Fewer workers = more stable on Windows laptops (8 workers + long runs often triggers flaky CUDA errors).
WORKERS = 2
# If training still crashes, try AMP = False (slower, sometimes more stable on laptop GPUs).
AMP = True
# After a crash: path to weights/last.pt and the same MODELS entry (e.g. "yolov8n.pt"). Others train normally.
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../dataset/data.yaml"
IMG_SIZE     = 640
EPOCHS       = 100
BATCH        = 32
DEVICE       = 0
PATIENCE     = 50
RESULTS_CSV  = "benchmark_yolov8.csv"
PERCLASS_CSV = "benchmark_yolov8_perclass.csv"

CLASS_NAMES = [
    'bench', 'bicycle', 'bus', 'bus_stop', 'car', 'crutch', 'curb', 'dog',
    'fire_hydrant', 'motorcycle', 'person', 'pole', 'spherical_roadblock',
    'stairs', 'stop_sign', 'street_light', 'traffic_light', 'train', 'tree',
    'truck', 'warning_column', 'waste_container'
]

# Comment out any variant you already finished to skip re-training.
MODELS = [
    "yolov8n.pt",
    "yolov8s.pt",
    "yolov8m.pt",
]


: 

In [ ]:
def benchmark_model(model_name):
    """Train a model on OOD and return benchmark metrics."""
    print(f"\n{'='*60}")
    print(f"  BENCHMARK: {model_name}")
    print(f"{'='*60}")

    resume_path = RESUME_CKPT
    use_resume = (
        resume_path
        and RESUME_MODEL == model_name
        and Path(resume_path).is_file()
    )
    
    if use_resume:
        print(f"  Resuming from {resume_path}")
        model = YOLO(resume_path)
        model.train(resume=True)
    else:
        if resume_path and RESUME_MODEL == model_name:
            print(f"  (RESUME_CKPT missing, training from scratch: {resume_path})")
        model = YOLO(model_name)
        model.train(
            data=DATA_YAML,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            device=DEVICE,
            workers=WORKERS,
            amp=AMP,
            optimizer='auto',  # Use best optimizer for the hardware
            cos_lr=True,       # Cosine LR scheduler
            lr0=0.01,          # Standard starting rate
            patience=PATIENCE,
            name=f"{model_name.replace('.pt', '')}_ood",
            save=True,
            plots=True,
        )

    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))

    metrics = best_model.val(
        data=DATA_YAML, split="test", device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS
    )

    # --- Inference speed benchmark ---
    test_img_dir = Path("../dataset/test/images")
    _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    test_images = [p for p in test_img_dir.glob("*.*") if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f"No test images under {test_img_dir.resolve()}")
    latencies = []
    # Warmup
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for img_path in test_images:
        t0 = time.perf_counter()
        best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)

    size_mb = best_path.stat().st_size / 1e6 if best_path.exists() else 0.0
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       round(float(metrics.box.map50), 4),
        "mAP@0.5:0.95":  round(float(metrics.box.map), 4),
        "precision":     round(p, 4),
        "recall":        round(r, 4),
        "F1":            round(f1, 4),
        "speed_ms/img":  round(float(np.mean(latencies)), 2),
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }

    # --- Per-class AP ---
    per_class = {}
    ap50_per_class = metrics.box.ap50
    for i, name in enumerate(CLASS_NAMES):
        if i < len(ap50_per_class):
            per_class[name] = round(float(ap50_per_class[i]), 4)

    del model, best_model
    gc.collect()
    _safe_cuda_empty_cache()

    return row, per_class


: 

In [ ]:
from pathlib import Path
root = Path("../runs/detect")
for variant in ["yolov8n", "yolov8s", "yolov8m"]:
    cands = list(root.glob(f"{variant}_ood*/weights/best.pt"))
    print(f"{variant}: {cands}")


yolov8n: [WindowsPath('../runs/detect/yolov8n_ood4/weights/best.pt')]
yolov8s: []
yolov8m: []


: 

In [ ]:
rows = []
all_per_class = {}

if RUN_TRAINING:
    print("RUN_TRAINING=True — full training for each entry in MODELS.\n")
    for model_name in MODELS:
        try:
            row, per_class = benchmark_model(model_name)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
            print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
            print(f"  Precision     : {row['precision']}")
            print(f"  Recall        : {row['recall']}")
            print(f"  F1            : {row['F1']}")
            print(f"  Speed         : {row['speed_ms/img']} ms/img")
            print(f"  Size          : {row['size_MB']} MB")
            print(f"  Params        : {row['params_M']} M")
        except Exception as e:
            print(f"  SKIPPED {model_name}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()
            print("  If CUDA errors persist: restart the kernel, set WORKERS=0 or AMP=False in the config cell, then rerun.")
else:
    print("RUN_TRAINING=False — building table from existing best.pt (no training).\n")
    for variant in BENCHMARK_VARIANTS:
        wp = _find_newest_best_pt(variant)
        if wp is None:
            print(f"  Skip {variant}: no ../runs/detect/{variant}_ood*/weights/best.pt")
            continue
        try:
            row, per_class = metrics_from_weights(wp, variant)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  {variant}: mAP@0.5={row['mAP@0.5']}  speed={row['speed_ms/img']} ms/img")
        except Exception as e:
            print(f"  SKIPPED {variant}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()

_cols = [
    "model", "mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1",
    "speed_ms/img", "size_MB", "params_M",
]
df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)
df.to_csv(RESULTS_CSV, index=False)

if all_per_class:
    df_pc = pd.DataFrame(all_per_class).T
else:
    df_pc = pd.DataFrame(columns=CLASS_NAMES)
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV} ({len(df)} row(s))")
print(f"Saved -> {PERCLASS_CSV}")


RUN_TRAINING=True — full training for each entry in MODELS.


  BENCHMARK: yolov8n.pt
New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=../dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0

KeyboardInterrupt: 

: 

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

_csv = Path(RESULTS_CSV)
if not _csv.is_file() or _csv.stat().st_size < 5:
    print("No results CSV yet. Run the **build benchmark** cell (with RUN_TRAINING False it only evaluates existing best.pt).")
else:
    df = pd.read_csv(RESULTS_CSV)
    if df.empty or "model" not in df.columns:
        print("benchmark_yolov8.csv is empty. Run training or add rows manually.")
    else:
        print("=" * 60)
        print("  YOLOv8 BENCHMARK — OOD Dataset (22 classes)")
        print("=" * 60)
        styled = (
            df.style
            .set_caption("YOLOv8 Benchmark — OOD Dataset")
            .format({
                "mAP@0.5":      "{:.4f}",
                "mAP@0.5:0.95": "{:.4f}",
                "precision":    "{:.4f}",
                "recall":       "{:.4f}",
                "F1":           "{:.4f}",
                "speed_ms/img": "{:.1f} ms",
                "size_MB":      "{:.1f} MB",
                "params_M":     "{:.1f} M",
            })
            .highlight_max(subset=["mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1"], color="#2d6a2e")
            .highlight_min(subset=["speed_ms/img", "size_MB", "params_M"], color="#1a5276")
            .set_properties(**{"text-align": "center", "font-size": "13px"})
            .set_table_styles([
                {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]},
                {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("padding", "8px")]},
            ])
            .hide(axis="index")
        )
        display(styled)


  YOLOv8 BENCHMARK — OOD Dataset (22 classes)


model,mAP@0.5,mAP@0.5:0.95,precision,recall,F1,speed_ms/img,size_MB,params_M
yolov8n,0.6291,0.4369,0.6901,0.5769,0.6284,59.4 ms,6.2 MB,3.0 M
yolov8s,0.6293,0.4473,0.6992,0.5889,0.6393,63.6 ms,22.5 MB,11.1 M
yolov8m,0.6499,0.4770,0.7028,0.5963,0.6452,34.2 ms,52.0 MB,25.9 M


: 

In [ ]:
from pathlib import Path

_pcsv = Path(PERCLASS_CSV)
if not _pcsv.is_file() or _pcsv.stat().st_size < 5:
    print("No per-class CSV yet. Run the **build benchmark** cell first.")
else:
    df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)
    if df_pc.empty:
        print("Per-class table is empty.")
    else:
        print("Per-class mAP@0.5 across YOLOv8 variants")
        print("-" * 60)
        styled_pc = (
            df_pc.style
            .set_caption("Per-Class mAP@0.5 — YOLOv8 Benchmark")
            .format("{:.4f}")
            .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
            .set_properties(**{"text-align": "center", "font-size": "12px"})
            .set_table_styles([
                {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
                {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
            ])
        )
        display(styled_pc)


Per-class mAP@0.5 across YOLOv8 variants
------------------------------------------------------------


,bench,bicycle,bus,bus_stop,car,crutch,curb,dog,fire_hydrant,motorcycle,person,pole,spherical_roadblock,stairs,stop_sign,street_light,traffic_light,train,tree,truck,warning_column,waste_container
model,,,,,,,,,,,,,,,,,,,,,,
yolov8n,0.5434,0.5453,0.8131,0.9462,0.6458,0.5986,0.4916,0.7029,0.8425,0.6208,0.2472,0.3439,0.9041,0.6157,0.9089,0.2476,0.5007,0.8295,0.1732,0.6277,0.8609,0.8299
yolov8s,0.5629,0.5702,0.7672,0.9378,0.6533,0.5284,0.4900,0.7242,0.8497,0.6837,0.2265,0.3534,0.9303,0.6832,0.9410,0.2990,0.4327,0.7249,0.1683,0.6224,0.8669,0.8276
yolov8m,0.6230,0.5898,0.8297,0.9582,0.6589,0.5567,0.5123,0.6999,0.8538,0.6478,0.2526,0.4165,0.9490,0.5758,0.9272,0.3344,0.4779,0.8715,0.1702,0.6594,0.8625,0.8712


: 

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

_csv = Path(RESULTS_CSV)
if not _csv.is_file() or _csv.stat().st_size < 5:
    print("No results CSV — run the build-benchmark cell first.")
else:
    df = pd.read_csv(RESULTS_CSV)
    if df.empty or "model" not in df.columns:
        print("Results CSV has no data rows — skip chart.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].barh(df["model"], df["mAP@0.5"], color="#2d6a2e")
        axes[0].set_xlabel("mAP@0.5")
        axes[0].set_title("Accuracy (mAP@0.5)")
        axes[0].set_xlim(0, 1)
        axes[1].barh(df["model"], df["speed_ms/img"], color="#1a5276")
        axes[1].set_xlabel("ms / image")
        axes[1].set_title("Inference Speed")
        axes[2].barh(df["model"], df["size_MB"], color="#c0392b")
        axes[2].set_xlabel("MB")
        axes[2].set_title("Model Size")
        plt.suptitle("YOLOv8 Benchmark — OOD Dataset", fontsize=16, fontweight="bold")
        plt.tight_layout()
        plt.savefig("benchmark_yolov8_chart.png", dpi=150, bbox_inches="tight")
        plt.show()


<Figure size 1800x500 with 3 Axes>

: 

In [ ]:
import pandas as pd
from ultralytics import YOLO
import matplotlib.pyplot as plt
from pathlib import Path
import random

_csv = Path(RESULTS_CSV)
if not _csv.is_file() or _csv.stat().st_size < 5:
    print("No results CSV — run the build-benchmark cell first.")
else:
    df_det = pd.read_csv(RESULTS_CSV)
    if df_det.empty or "mAP@0.5" not in df_det.columns:
        print("Results CSV has no data rows.")
    else:
        best_model_name = str(df_det.loc[df_det["mAP@0.5"].idxmax(), "model"])
        run_root = Path("../runs/detect")
        candidates = list(run_root.glob(f"{best_model_name}_ood*/weights/best.pt"))
        if not candidates:
            print(f"No weights found matching ../runs/detect/{best_model_name}_ood*/weights/best.pt")
        else:
            best_path = max(candidates, key=lambda p: p.stat().st_mtime)
            print(f"Best model: {best_model_name} — loading {best_path}")
            model = YOLO(str(best_path))
            test_img_dir = Path("../dataset/test/images")
            _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            all_imgs = [f.name for f in test_img_dir.iterdir() if f.suffix.lower() in _ext]
            if not all_imgs:
                print(f"No images in {test_img_dir.resolve()}")
            else:
                images = random.sample(all_imgs, min(9, len(all_imgs)))
                fig, axes = plt.subplots(3, 3, figsize=(18, 18))
                for ax, img_name in zip(axes.flatten(), images):
                    img_path = test_img_dir / img_name
                    results = model(str(img_path), conf=0.25, verbose=False)[0]
                    img_plot = results.plot()[:, :, ::-1]
                    ax.imshow(img_plot)
                    ax.axis("off")
                    ax.set_title(f"{len(results.boxes)} detections", fontsize=10)
                plt.suptitle(f"{best_model_name} (Best) — Detections on OOD Test Set", fontsize=16, y=1.01)
                plt.tight_layout()
                plt.savefig("benchmark_yolov8_detections.png", dpi=150, bbox_inches="tight")
                plt.show()


Best model: yolov8m — loading ..\runs\detect\yolov8m_ood\weights\best.pt


<Figure size 1800x1800 with 9 Axes>

: 